In [ ]:
import pandas as pd
df = pd.read_csv("/content/Maternal Health Risk Data Set.xls")
df.info()
df.describe()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   object 
dtypes: float64(2), int64(4), object(1)
memory usage: 55.6+ KB


,0
Age,0
SystolicBP,0
DiastolicBP,0
BS,0
BodyTemp,0
HeartRate,0
RiskLevel,0


In [ ]:
df["RiskLevel"].value_counts()

,count
RiskLevel,
low risk,406
mid risk,336
high risk,272


In [ ]:
df.describe()

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
count,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000
mean,29.871795,113.198225,76.460552,8.725986,98.665089,74.301775
std,13.474386,18.403913,13.885796,3.293532,1.371384,8.088702
min,10.000000,70.000000,49.000000,6.000000,98.000000,7.000000
25%,19.000000,100.000000,65.000000,6.900000,98.000000,70.000000
50%,26.000000,120.000000,80.000000,7.500000,98.000000,76.000000
75%,39.000000,120.000000,90.000000,8.000000,98.000000,80.000000
max,70.000000,160.000000,100.000000,19.000000,103.000000,90.000000


In [ ]:
df['Age'].mean()
df[(df["Age"] < 16) | (df["Age"] > 50)] # Too many cases out of typical reprodactive range.

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
15,15,120,80,7.01,98.0,70,low risk
19,10,70,50,6.90,98.0,70,low risk
35,12,95,60,6.10,102.0,60,low risk
36,60,120,80,6.10,98.0,75,low risk
37,55,100,65,6.10,98.0,66,low risk
...,...,...,...,...,...,...,...
1000,12,90,60,11.00,102.0,60,high risk
1001,60,120,85,15.00,98.0,60,high risk
1002,55,140,95,19.00,98.0,77,high risk
1004,13,90,65,9.00,101.0,80,high risk


In [ ]:
df.describe()
df[df["HeartRate"] <= 40] # I put 40 because less then 40 is not phisiologically possible

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
499,16,120,75,7.9,98.0,7,low risk
908,16,120,75,7.9,98.0,7,low risk


In [ ]:
df = df[df["HeartRate"] >= 40]
df.shape

(1012, 7)

In [ ]:
df.duplicated().sum() # More then a half of the data are duplicates; risk of data leakage when teaching a model.
df = df.drop_duplicates()
df.shape
df["RiskLevel"].value_counts()

,count
RiskLevel,
low risk,233
high risk,112
mid risk,106


In [ ]:
# Start of teaching the model

from sklearn.model_selection import train_test_split

# X = the 6 columns with measurements (what the model sees)
X = df.drop(columns=["RiskLevel"])

# y = the column we want to predict
y = df["RiskLevel"]

# Split the rows: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% of the rows go to the test set
    random_state=42,    # so the split is the same every time
    stratify=y          # keeps the proportion of risk groups
)

print(X_train.shape, X_test.shape) # 360 rows out of 561 is for learning and 91 rows is for testing. 6 columns in both batches.

(360, 6) (91, 6)


In [ ]:
# Share of each risk group in train and in test
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

RiskLevel
low risk     0.516667
high risk    0.247222
mid risk     0.236111
Name: proportion, dtype: float64
RiskLevel
low risk     0.516484
high risk    0.252747
mid risk     0.230769
Name: proportion, dtype: float64


In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Create the model (max_iter gives it enough steps to finish learning)
model = LogisticRegression(max_iter=5000)

# 2. Train: the model sees the measurements AND the correct answers
model.fit(X_train, y_train)

# 3. Predict: the model sees only the measurements of the test patients
y_pred = model.predict(X_test)

# Look at the first 10 predictions
print(y_pred[:10])

['low risk' 'low risk' 'low risk' 'low risk' 'low risk' 'mid risk'
 'high risk' 'low risk' 'low risk' 'low risk']


In [ ]:
from sklearn.metrics import accuracy_score

# Compare the correct answers with the model's predictions
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.6703296703296703


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Quality metrics for each risk group separately
print(classification_report(y_test, y_pred))

# Table of errors: rows = real group, columns = predicted group
labels = ["low risk", "mid risk", "high risk"]
print(confusion_matrix(y_test, y_pred, labels=labels))

              precision    recall  f1-score   support

   high risk       0.81      0.57      0.67        23
    low risk       0.64      1.00      0.78        47
    mid risk       1.00      0.05      0.09        21

    accuracy                           0.67        91
   macro avg       0.82      0.54      0.51        91
weighted avg       0.76      0.67      0.59        91

[[47  0  0]
 [17  1  3]
 [10  0 13]]


In [ ]:
# Let's try a Random forest, because in the regression the model misses 43% cases of high-risk.

from sklearn.ensemble import RandomForestClassifier

# 1. Create the model (random_state makes the result reproducible).
rf_model = RandomForestClassifier(random_state=42)

# 2. Train: the model sees the measurements AND the correct answers
rf_model.fit(X_train, y_train)

# 3. Predict: the model sees only the measurements of the test patients
y_pred_rf = rf_model.predict(X_test)

# Look at the first 10 predictions
print(y_pred_rf[:10])

['low risk' 'low risk' 'low risk' 'mid risk' 'mid risk' 'mid risk'
 'high risk' 'low risk' 'low risk' 'low risk']


In [ ]:
# Quality metrics and error table for Random Forest
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf, labels=labels))

              precision    recall  f1-score   support

   high risk       0.89      0.70      0.78        23
    low risk       0.61      0.74      0.67        47
    mid risk       0.19      0.14      0.16        21

    accuracy                           0.59        91
   macro avg       0.56      0.53      0.54        91
weighted avg       0.59      0.59      0.58        91

[[35 12  0]
 [16  3  2]
 [ 6  1 16]]


In [ ]:
# Rows where all 6 measurements are identical but the risk label differs
features = ["Age", "SystolicBP", "DiastolicBP", "BS", "BodyTemp", "HeartRate"]
print(df.duplicated(subset=features, keep=False).sum())

71


So, 71 ot of 451 is 16% of the table that has duplicated data but in another risk group. The boundary between low and mid risk is blurry in this data. The risk level was probably assigned using information that is not in the table (for example, medical history), or different clinicians rated similar cases differently. This is why mid risk is not recognised. The model is not the weak point here: the measurements themselves do not contain the information needed to tell the two groups apart. High risk separates much more cleanly, with only a few conflicting cases. So, probably it's label inconsistancy

In [ ]:
# How much each measurement contributed to the model's decisions
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

BS             0.368760
SystolicBP     0.181054
Age            0.146580
DiastolicBP    0.108637
HeartRate      0.106038
BodyTemp       0.088931
dtype: float64


The ML part of the project is mostly complete:

the data is cleaned: 2 impossible values and 561 duplicates removed;
two data limitations are identified: age and inconsistent labels;
two models are trained and compared;
the metric is chosen, recall for high risk, and the choice is explained;
the main risk factors are known.

In [42]:
results = X_test.copy()
results["actual_risk"] = y_test
results["predicted_risk"] = y_pred_rf

# True if the model's prediction matches the real risk level
results["correct"] = results["actual_risk"] == results["predicted_risk"]
print(results.head())
results.to_csv("test_predictions.csv", index=False)

     Age  SystolicBP  DiastolicBP   BS  BodyTemp  HeartRate actual_risk  \
310   55         120           90  6.9      98.0         76    low risk   
615   34         110           70  7.0      98.0         80   high risk   
386   30         120           80  7.8      98.0         70    low risk   
704   12         100           50  6.0      98.0         70    mid risk   
4     35         120           60  6.1      98.0         76    low risk   

    predicted_risk  correct  
310       low risk     True  
615       low risk    False  
386       low risk     True  
704       mid risk     True  
4         mid risk    False  
